In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
from alphabase.peptide.fragment import get_charged_frag_types
import pandas as pd

fasta_list = [
    r"D:\DIANN-beg\uniprot_reviewed_human_canonical_and_isoforms_20230330.fasta"
]
# output spectral library in hdf format
hdf_path = r'D:\optimising-plasma-proteomics\alphapept\alphapeptdeep\output\output-lib.speclib.hdf'

protease="trypsin"
nce = 30
instrument = 'timsTOF'

add_phos=False

protease_dict = {
    "trypsin": "([KR])", # this is in fact the "trypsin/P"
    "lysc": "([K])",
    "lysn": r"\w(?=K)",
}
min_pep_len = 7
max_pep_len = 41
max_miss_cleave = 2
max_var_mods = 2
min_pep_mz = 300
max_pep_mz = 1200
precursor_charge_min = 2
precursor_charge_max = 4

var_mods = []
var_mods += ['Acetyl@Protein_N-term', 'Oxidation@M']
#var_mods += ['Phospho@S','Phospho@T','Phospho@Y']


frag_types = get_charged_frag_types(
    ['b','y']+
    (['b_modloss','y_modloss'] if add_phos else []),
    2
)

In [3]:
digest = protease_dict[protease]

In [4]:
from peptdeep.protein.fasta import PredictSpecLibFasta
from peptdeep.pretrained_models import ModelManager

model_mgr = ModelManager(device='gpu')

model_mgr.nce = nce
model_mgr.instrument = instrument

fasta_lib = PredictSpecLibFasta(
    model_mgr,
    protease=digest,
    charged_frag_types=frag_types,
    var_mods=var_mods,
    fix_mods=['Carbamidomethyl@C'],
    max_missed_cleavages=max_miss_cleave,
    max_var_mod_num=max_var_mods,
    peptide_length_max=max_pep_len,
    peptide_length_min=min_pep_len,
    precursor_charge_min=precursor_charge_min,
    precursor_charge_max=precursor_charge_max,
    precursor_mz_min=min_pep_mz,
    precursor_mz_max=max_pep_mz,
    decoy=None
)

d:\optimising-plasma-proteomics\alphapept\alphapeptdeep\peptdeep-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\optimising-plasma-proteomics\alphapept\alphapeptdeep\peptdeep-venv\Lib\site-packages\peptdeep\model\ms2.py:416: UserWarning: mask_modloss is deprecated and will be removed in the future. To mask the modloss fragments, the charged_frag_types should not include the modloss fragments.
  warnings.warn(


In [5]:
fasta_lib.get_peptides_from_fasta_list(fasta_list)

In [6]:
fasta_lib.append_decoy_sequence()
fasta_lib.add_modifications()

In [ ]:
fasta_lib.precursor_df['nAA'] = fasta_lib.precursor_df.sequence.str.len()
fasta_lib.precursor_df.sort_values('nAA', inplace=True)
fasta_lib.precursor_df.reset_index(drop=True, inplace=True)

In [ ]:
fasta_lib.add_charge()

In [ ]:
fasta_lib.hash_precursor_df()
fasta_lib.calc_precursor_mz()
fasta_lib.precursor_df

,sequence,protein_idxes,miss_cleavage,is_prot_nterm,is_prot_cterm,mods,mod_sites,nAA,charge,mod_seq_hash,mod_seq_charge_hash,precursor_mz
0,KQSHDNR,9816;40869,1,False,False,,,7,2,12535707109762685139,12535707109762685141,442.720290
1,KQSHDNR,9816;40869,1,False,False,,,7,3,12535707109762685139,12535707109762685142,295.482619
2,KQSHDNR,9816;40869,1,False,False,,,7,4,12535707109762685139,12535707109762685143,221.863783
3,SPEGSRK,2040;2041;18519;38563,1,False,False,,,7,2,4067122071945261943,4067122071945261945,380.701035
4,SPEGSRK,2040;2041;18519;38563,1,False,False,,,7,3,4067122071945261943,4067122071945261946,254.136449
...,...,...,...,...,...,...,...,...,...,...,...,...
12256999,MHVPAPKLALPGHAESYNPPPEYLLSEEERLAWEQQEPGER,6431;39280,2,False,False,Oxidation@M,1,41,3,10393568852829757155,10393568852829757158,1571.101040
12257000,MHVPAPKLALPGHAESYNPPPEYLLSEEERLAWEQQEPGER,6431;39280,2,False,False,Oxidation@M,1,41,4,10393568852829757155,10393568852829757159,1178.577599
12257001,ELSFIVNSSVFLEEVISELLCKILYAFSHNMLVTENPDRVK,7380,2,False,False,Carbamidomethyl@C,21,41,2,3221513280197216708,3221513280197216710,2392.239902
12257002,ELSFIVNSSVFLEEVISELLCKILYAFSHNMLVTENPDRVK,7380,2,False,False,Carbamidomethyl@C,21,41,3,3221513280197216708,3221513280197216711,1595.162360


In [ ]:
fasta_lib.precursor_df.drop(fasta_lib.precursor_df.query('`precursor_mz` < 300 or `precursor_mz` > 1200').index, inplace=True)
fasta_lib.precursor_df.reset_index(drop=True, inplace=True)

In [ ]:
fasta_lib.precursor_df['instrument'] = model_mgr.instrument
fasta_lib.precursor_df['nce'] = model_mgr.nce
res = fasta_lib.model_manager.predict_all(
    fasta_lib.precursor_df,
    predict_items=['rt','mobility','ms2'],
    frag_types = frag_types,
)
fasta_lib.set_precursor_and_fragment(
    **res
)

2025-11-07 18:00:04> Predicting RT ...


100%|██████████| 35/35 [06:43<00:00, 11.52s/it]


2025-11-07 18:06:48> Predicting mobility ...


100%|██████████| 35/35 [08:11<00:00, 14.05s/it]


2025-11-07 18:16:49> Predicting MS2 ...


100%|██████████| 35/35 [18:17<00:00, 31.37s/it]


In [ ]:
import os, psutil
import numpy as np
process = psutil.Process(os.getpid())
print(f'{len(fasta_lib.precursor_df)*1e-6:.2f}M precursors with {np.prod(fasta_lib.fragment_mz_df.values.shape, dtype=float)*(1e-6):.2f}M fragments used {process.memory_info().rss/1024**3:.4f} GB memory')

9.00M precursors with 630.34M fragments used 4.7758 GB memory


In [ ]:
fasta_lib.translate_rt_to_irt_pred()


Predict RT for 11 iRT precursors.
Linear regression of `rt_pred` to `irt`:
   R_square         R       slope  intercept  test_num
0  0.990063  0.995019  152.232936 -39.232042        11


,sequence,protein_idxes,miss_cleavage,is_prot_nterm,is_prot_cterm,mods,mod_sites,nAA,charge,mod_seq_hash,...,precursor_mz,instrument,nce,rt_pred,rt_norm_pred,ccs_pred,mobility_pred,frag_start_idx,frag_stop_idx,irt_pred
0,KQSHDNR,9816;40869,1,False,False,,,7,2,12535707109762685139,...,442.720290,timsTOF,30,0.076455,0.076455,308.764832,0.759041,0,6,-27.593109
1,SPEGSRK,2040;2041;18519;38563,1,False,False,,,7,2,4067122071945261943,...,380.701035,timsTOF,30,0.029027,0.029027,301.090668,0.738334,6,12,-34.813165
2,SDSGSRR,23085,1,False,False,,,7,2,3357315649186943522,...,382.685916,timsTOF,30,0.012741,0.012741,297.333496,0.729188,12,18,-37.292415
3,NPARTCR,2040;2041;2370;18519;38563,1,False,False,Carbamidomethyl@C,6,7,2,9547902782194548795,...,437.719236,timsTOF,30,0.000000,0.000000,308.071899,0.757205,18,24,-39.232042
4,LFIFLGK,9324;32080,0,False,False,,,7,2,16433146202609194560,...,419.265282,timsTOF,30,0.829399,0.829399,310.174713,0.761854,24,30,87.029829
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8998332,VEMGFCHVGQNGFELLTSSYLPASASQSAEIIAVRIEISRK,32597,2,False,False,Oxidation@M;Carbamidomethyl@C,3;6,41,4,2857143767610083348,...,1128.572894,timsTOF,30,0.874504,0.874504,969.969788,1.207212,157584885,157584925,93.896238
8998333,VEMGFCHVGQNGFELLTSSYLPASASQSAEIIAVRIEISRK,32597,2,False,False,Carbamidomethyl@C,6,41,4,1622887496054026454,...,1124.574165,timsTOF,30,0.877959,0.877959,980.098022,1.219804,157584925,157584965,94.422227
8998334,PGCIIELDAITNQCGSNTQLLHVFQEDFIIGYKPHKEDMEK,9930;32964;32965,2,False,False,Carbamidomethyl@C;Carbamidomethyl@C,3;14,41,4,14290545395395826800,...,1197.833114,timsTOF,30,0.869339,0.869339,945.694214,1.177208,157584965,157585005,93.110030
8998335,MHVPAPKLALPGHAESYNPPPEYLLSEEERLAWEQQEPGER,6431;39280,2,False,False,Oxidation@M,1,41,4,10393568852829757155,...,1178.577599,timsTOF,30,0.735830,0.735830,841.294556,1.047201,157585005,157585045,72.785475


In [ ]:
fasta_lib.save_hdf(hdf_path)

In [ ]:
import pyarrow as pa
from pyarrow import parquet as pq
parquet_file = pq.ParquetFile(r"D:\DIANN-beg\outputs\trial-lib-to-parquet\report-lib.parquet")
df = parquet_file.read(columns=['Precursor.Id', 'Decoy']).to_pandas().drop_duplicates(subset=['Precursor.Id'])
len(df)

9117643

In [ ]:
parquet_file.schema_arrow

Precursor.Id: string not null
Modified.Sequence: string not null
Stripped.Sequence: string not null
Precursor.Charge: int64 not null
Proteotypic: int64 not null
Decoy: int64 not null
N.Term: int64 not null
C.Term: int64 not null
RT: float not null
IM: float not null
Q.Value: float not null
Peptidoform.Q.Value: float not null
PTM.Site.Confidence: float not null
PG.Q.Value: float not null
Precursor.Mz: float not null
Product.Mz: float not null
Relative.Intensity: float not null
Fragment.Type: string not null
Fragment.Charge: int64 not null
Fragment.Series.Number: int64 not null
Fragment.Loss.Type: string not null
Exclude.From.Quant: int64 not null
Protein.Ids: string not null
Protein.Group: string not null
Protein.Names: string not null
Genes: string not null
Flags: int64 not null